# BYV-AI Training: BYV Markup v0.1 + Qwen3-4B-Instruct-2507

**Base model:** `Qwen/Qwen3-4B-Instruct-2507` (pinned revision `cdbee75f17c01a7cc42f958dc650907174af0554`)  
**Dataset:** BYV Markup v0.1 (280 train / 35 eval; held-out test is not opened here)  
**Method:** supervised fine-tuning (SFT) + 4-bit NF4 QLoRA  
**Target hardware:** Google Colab T4 as a feasibility run (FP16, batch 1, gradient checkpointing)

Notebook ini melatih model **text-only** untuk tiga mode output: TEXT, MIXED, dan STRUCTURED. BYV tetap dihasilkan sebagai teks biasa. Notebook tidak menjalankan tool dan tidak menghubungkan model ke agent atau SDK.

System prompt lengkap diambil dari folder `system/` di repo pada commit yang dipin dan diverifikasi dengan Git blob SHA. Dataset menyimpan schema tools per contoh, lalu notebook menyisipkan prompt yang sama untuk train/eval.

> **Catatan kualitas:** dataset ini migrasi format mekanis dari seed sintetis BYV-FP v0.2.1. Split dipertahankan, tool arguments divalidasi terhadap schema sumber, dan mode MIXED diberi lead-in singkat otomatis. Ini belum human-reviewed semantic ground truth dan bukan benchmark kualitas umum.

Jalankan semua cell dari atas ke bawah. Jika T4 OOM atau terlalu banyak sampel melewati batas panjang, gunakan GPU dengan VRAM lebih besar. Jangan memotong system prompt secara diam-diam.

In [ ]:
%pip install -q -U "transformers>=4.51.0,<5.0.0" "accelerate>=1.0.0,<2.0.0" "datasets>=3.0.0,<5.0.0" "peft>=0.14.0,<1.0.0" "bitsandbytes>=0.45.0,<1.0.0" "safetensors>=0.4.5" "requests>=2.31.0" "packaging>=23.0"


In [ ]:
import os, sys, json, re, time, math, hashlib, random, inspect, platform
from pathlib import Path
from collections import Counter

import requests
import torch
import transformers
from transformers import AutoTokenizer
from datasets import Dataset
from packaging.version import Version

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if Version(torch.__version__.split('+')[0]) < Version("2.6.0"):
    raise RuntimeError("Qwen3-2507 recommends PyTorch >= 2.6.0. Select a Colab runtime with a newer CUDA-enabled PyTorch build.")


In [ ]:
# ---------- Experiment config ----------
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"
DATASET_VERSION = "byv-markup-v0.1"
ZIP_NAME = "BYV-AI_dataset_BYV-Markup_v0.1.zip"
ZIP_ROOT = "BYV-AI_dataset_byv_markup_v0.1/"
EXPECTED_DATA_SHA256 = {
    "train.jsonl": "bc2755b32db2c3f8886be6dfa3550cb440955ca8a517027c2b04b586a0798dcc",
    "eval.jsonl": "16fc5f603020277a2f45bf8038c270c4cd9824360b47e9dba977e8983dc761f4"
}

PROMPT_COMMIT = "0929d831b81a9e238ac27428310ce14503357190"
PROMPT_URL = "https://raw.githubusercontent.com/naufalelghani/byv-ai/0929d831b81a9e238ac27428310ce14503357190/system/BYV_SYSTEM_PROMPT_TEMPLATE_v0.1.md"
EXPECTED_PROMPT_GIT_BLOB_SHA = "c7e2ee8d06cae49518b084461ccfbdd9b1434374"

SEED = 42
MAX_LENGTH = 4096  # full system prompt is retained; overlength samples are reported, never silently truncated
NUM_EPOCHS = 2
LEARNING_RATE = 1.0e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_BASELINE_NEW_TOKENS = 768
MIN_TRAIN_RECORDS = 200
MIN_TRAIN_TURNS = 150
MAX_ALLOWED_SKIP_RATIO = 0.20

RUN_NAME = "byv-ai-qwen3-4b-instruct-2507-byv-markup-qlora-t4-fp16-v0.1"
SYSTEM_SENTINEL = "__BYV_SYSTEM_PROMPT_TEMPLATE_INJECTED_AT_RUNTIME__"
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Model:", MODEL_ID, "@", MODEL_REVISION)
print("Dataset:", DATASET_VERSION)
print("Run:", RUN_NAME)


## 1. Mount Google Drive dan siapkan dataset

Upload ZIP **hanya pada pemakaian pertama**. File ZIP akan disimpan di Google Drive. Run berikutnya dan notebook evaluasi dapat memakai salinan yang sama tanpa upload ulang.

Cell ini hanya membaca `train.jsonl` dan `eval.jsonl`. `test.jsonl` tidak dibuka oleh notebook training.

In [ ]:
from google.colab import drive, files
import io, zipfile

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/BYV-AI')
ARCHIVE_DIR = DRIVE_ROOT / 'datasets'
DATA_DIR = DRIVE_ROOT / 'data' / DATASET_VERSION
OUTPUT_DIR = DRIVE_ROOT / 'outputs' / RUN_NAME
EXPERIMENT_DIR = DRIVE_ROOT / 'experiments' / RUN_NAME
for folder in (ARCHIVE_DIR, DATA_DIR, OUTPUT_DIR, EXPERIMENT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

ARCHIVE_PATH = ARCHIVE_DIR / ZIP_NAME
if not ARCHIVE_PATH.exists():
    print(f"Upload {ZIP_NAME} once. It will be saved to Google Drive.")
    uploaded = files.upload()
    if ZIP_NAME not in uploaded:
        raise FileNotFoundError(f"Please upload the exact file: {ZIP_NAME}")
    ARCHIVE_PATH.write_bytes(uploaded[ZIP_NAME])
    del uploaded
else:
    print("Using dataset archive already saved in Drive:", ARCHIVE_PATH)

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    members = set(archive.namelist())
    for filename, expected_hash in EXPECTED_DATA_SHA256.items():
        member = ZIP_ROOT + filename
        if member not in members:
            raise FileNotFoundError(f"Dataset ZIP is missing {member}")
        payload = archive.read(member)  # train/eval only; test.jsonl is deliberately never opened
        actual_hash = hashlib.sha256(payload).hexdigest()
        if actual_hash != expected_hash:
            raise RuntimeError(f"SHA-256 mismatch for {filename}: {actual_hash} != {expected_hash}")
        (DATA_DIR / filename).write_bytes(payload)
        print(f"Verified {filename}: {actual_hash}")

TRAIN_PATH = DATA_DIR / 'train.jsonl'
EVAL_PATH = DATA_DIR / 'eval.jsonl'
print("Train:", TRAIN_PATH)
print("Eval:", EVAL_PATH)
print("Adapter output:", OUTPUT_DIR)


## 2. Ambil system prompt kanonis dari repo

Notebook mengambil template dari commit repo yang dipin, lalu memverifikasi Git blob SHA. Dengan cara ini, fine-tuning tidak memakai salinan system prompt yang tidak diketahui versinya.

In [ ]:
response = requests.get(PROMPT_URL, timeout=60)
response.raise_for_status()
prompt_bytes = response.content
blob_header = f"blob {len(prompt_bytes)}\0".encode("utf-8")
actual_prompt_git_sha = hashlib.sha1(blob_header + prompt_bytes).hexdigest()
if actual_prompt_git_sha != EXPECTED_PROMPT_GIT_BLOB_SHA:
    raise RuntimeError(
        f"System prompt template SHA mismatch: {actual_prompt_git_sha} != {EXPECTED_PROMPT_GIT_BLOB_SHA}"
    )
SYSTEM_TEMPLATE = prompt_bytes.decode("utf-8")
if SYSTEM_TEMPLATE.count("{{AVAILABLE_TOOLS}}") != 1:
    raise RuntimeError("System prompt template must contain exactly one {{AVAILABLE_TOOLS}} placeholder")


def build_system_prompt(tools):
    if not tools:
        tools_text = "No tools are available for this request."
    else:
        tools_text = (
            "Trusted host-provided tool definitions and exact input schemas:\n"
            + json.dumps(tools, ensure_ascii=False, indent=2)
        )
    return SYSTEM_TEMPLATE.replace("{{AVAILABLE_TOOLS}}", tools_text)

print("System prompt loaded and verified.")
print("Template Git blob:", actual_prompt_git_sha)
print("Fixed template characters:", len(SYSTEM_TEMPLATE))


## 3. Validasi split dan mode output BYV

Validasi ini memeriksa struktur contoh, mode output, jumlah blok, tool name, serta pemisahan train/eval. Validitas sintaks tidak membuktikan bahwa jawaban benar secara semantik.

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"{path}:{line_no}: invalid JSON: {exc}") from exc
    return rows

CALLS_OPEN_RE = re.compile(r"(?<!\\)<\|BYV\| calls>")
INVOKE_OPEN_RE = re.compile(r'<\|BYV\| invoke name="([A-Za-z_][A-Za-z0-9_.:-]*)" id="([A-Za-z0-9][A-Za-z0-9._:-]{0,63})">')


def classify_mode(text):
    match = CALLS_OPEN_RE.search(text)
    if not match:
        return "TEXT"
    before = text[:match.start()].strip()
    after_start = match.end()
    close = text.find("</|BYV| calls>", after_start)
    if close < 0:
        return "INVALID"
    after = text[close + len("</|BYV| calls>"):].strip()
    return "MIXED" if before or after else "STRUCTURED"


def validate_records(rows, label):
    ids = [r.get("id") for r in rows]
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate record IDs in {label}")
    mode_counts = Counter()
    assistant_turns = 0
    for row in rows:
        if not isinstance(row.get("messages"), list) or not row["messages"]:
            raise ValueError(f"{label}/{row.get('id')}: missing messages")
        if row["messages"][0].get("role") != "system" or row["messages"][0].get("content") != SYSTEM_SENTINEL:
            raise ValueError(f"{label}/{row['id']}: system prompt sentinel missing")
        tools = row.get("tools", [])
        if not isinstance(tools, list):
            raise ValueError(f"{label}/{row['id']}: tools must be a list")
        tool_names = {t.get("name") for t in tools}
        seen_call_ids = set()
        found_modes = []
        for i, msg in enumerate(row["messages"]):
            if msg.get("role") not in {"system", "user", "assistant"}:
                raise ValueError(f"{label}/{row['id']}: unsupported role {msg.get('role')}")
            content = msg.get("content", "")
            if "BYV-FP/1.0" in content or "DSML" in content or "｜DSML｜" in content:
                raise ValueError(f"{label}/{row['id']}: legacy protocol text remains in messages")
            if msg["role"] == "assistant":
                assistant_turns += 1
                mode = classify_mode(content)
                if mode == "INVALID":
                    raise ValueError(f"{label}/{row['id']} assistant[{i}]: incomplete calls block")
                if mode in {"MIXED", "STRUCTURED"}:
                    if content.count("</|BYV| calls>") != 1:
                        raise ValueError(f"{label}/{row['id']} assistant[{i}]: calls closing tag count invalid")
                    invoke_matches = list(INVOKE_OPEN_RE.finditer(content))
                    if not invoke_matches:
                        raise ValueError(f"{label}/{row['id']} assistant[{i}]: no invoke blocks")
                    call_ids = [m.group(2) for m in invoke_matches]
                    if len(call_ids) != len(set(call_ids)):
                        raise ValueError(f"{label}/{row['id']} assistant[{i}]: duplicate call IDs")
                    for tool_name, _ in [m.groups() for m in invoke_matches]:
                        if tool_name not in tool_names:
                            raise ValueError(f"{label}/{row['id']} assistant[{i}]: unknown tool {tool_name}")
                elif not content.strip():
                    raise ValueError(f"{label}/{row['id']} assistant[{i}]: empty TEXT target")
                found_modes.append(mode)
                mode_counts[mode] += 1
        if found_modes != row.get("assistant_output_modes"):
            raise ValueError(f"{label}/{row['id']}: assistant_output_modes does not match messages")
    return mode_counts, assistant_turns

TRAIN_ROWS = load_jsonl(TRAIN_PATH)
EVAL_ROWS = load_jsonl(EVAL_PATH)
train_modes, train_turns = validate_records(TRAIN_ROWS, "train")
eval_modes, eval_turns = validate_records(EVAL_ROWS, "eval")
if len(TRAIN_ROWS) < MIN_TRAIN_RECORDS:
    raise RuntimeError(f"Expected at least {MIN_TRAIN_RECORDS} training records; got {len(TRAIN_ROWS)}")
if {r['id'] for r in TRAIN_ROWS} & {r['id'] for r in EVAL_ROWS}:
    raise RuntimeError("Train/eval ID overlap detected")

print("Train records:", len(TRAIN_ROWS), "assistant turns:", train_turns, "modes:", dict(train_modes))
print("Eval records:", len(EVAL_ROWS), "assistant turns:", eval_turns, "modes:", dict(eval_modes))
print("Dataset and mode validation passed. Held-out test split was not loaded.")


## 4. Periksa GPU dan dtype

Tesla T4 memakai FP16. Notebook memilih BF16 hanya jika GPU benar-benar mendukungnya.

In [ ]:
GPU_AVAILABLE = torch.cuda.is_available()
GPU_NAME = None
GPU_VRAM_GB = None
GPU_COMPUTE_CAPABILITY = None
if GPU_AVAILABLE:
    props = torch.cuda.get_device_properties(0)
    GPU_NAME = props.name
    GPU_VRAM_GB = round(props.total_memory / (1024**3), 2)
    GPU_COMPUTE_CAPABILITY = torch.cuda.get_device_capability(0)
    native_bf16 = GPU_COMPUTE_CAPABILITY[0] >= 8 and torch.cuda.is_bf16_supported()
    COMPUTE_DTYPE = torch.bfloat16 if native_bf16 else torch.float16
    print("GPU:", GPU_NAME)
    print("VRAM:", GPU_VRAM_GB, "GB")
    print("Compute capability:", GPU_COMPUTE_CAPABILITY)
    print("Compute dtype:", COMPUTE_DTYPE)
else:
    COMPUTE_DTYPE = torch.float16
    print("No CUDA GPU. Dataset validation is available, but QLoRA training will be skipped.")


## 5. Tokenizer dan chat-template probe

Qwen3-4B-Instruct-2507 adalah model non-thinking. Tidak ada blok `<think>` yang perlu ditambahkan atau dipotong. Model dilatih dengan target assistant apa adanya.

In [ ]:
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_fast=True)
if TOKENIZER.pad_token_id is None:
    TOKENIZER.pad_token = TOKENIZER.eos_token
TOKENIZER.padding_side = "right"

probe_messages = [
    {"role": "system", "content": "You are BYV-AI. Answer in plain text."},
    {"role": "user", "content": "Jelaskan fungsi README dalam satu kalimat."},
]
probe_text = TOKENIZER.apply_chat_template(
    probe_messages, tokenize=False, add_generation_prompt=True
)
print("Chat-template suffix:", repr(probe_text[-100:]))
if "<think>" in probe_text:
    print("Warning: unexpected think marker found in chat template; inspect tokenizer configuration.")
else:
    print("Non-thinking chat-template probe passed.")


## 6. Expand assistant turns dan terapkan assistant-only loss

Setiap jawaban assistant menjadi satu sample causal-LM. Token prompt (termasuk system prompt lengkap dan tool schema) diberi label `-100`; loss hanya dihitung pada target assistant. Riwayat multi-turn tetap dipertahankan.

Sample yang melebihi `MAX_LENGTH` dilewati, bukan dipotong sehingga system prompt atau target hilang.

In [ ]:
def messages_with_system_prompt(row):
    messages = [dict(m) for m in row["messages"]]
    messages[0]["content"] = build_system_prompt(row.get("tools", []))
    return messages


def token_ids(messages, add_generation_prompt):
    result = TOKENIZER.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=add_generation_prompt,
    )
    if isinstance(result, dict):
        result = result["input_ids"]
    if isinstance(result, torch.Tensor):
        result = result.tolist()
    return list(result)


def expand_assistant_turns(rows, split_name):
    samples = []
    skipped_long = []
    lengths = []
    for row in rows:
        messages = messages_with_system_prompt(row)
        for i, msg in enumerate(messages):
            if msg["role"] != "assistant":
                continue
            prefix_messages = messages[:i]
            if not any(m["role"] == "user" for m in prefix_messages):
                continue
            target_content = msg["content"]
            full_messages = prefix_messages + [{"role": "assistant", "content": target_content}]
            prompt_ids = token_ids(prefix_messages, add_generation_prompt=True)
            full_ids = token_ids(full_messages, add_generation_prompt=False)
            if full_ids[:len(prompt_ids)] != prompt_ids:
                raise RuntimeError(
                    f"Chat-template prefix mismatch for {split_name}/{row['id']} assistant[{i}]."
                )
            target_ids = full_ids[len(prompt_ids):]
            lengths.append(len(full_ids))
            if len(target_ids) == 0 or len(full_ids) > MAX_LENGTH:
                skipped_long.append({
                    "id": row["id"], "assistant_index": i,
                    "total_tokens": len(full_ids), "target_tokens": len(target_ids),
                })
                continue
            samples.append({
                "input_ids": full_ids,
                "labels": [-100] * len(prompt_ids) + target_ids,
                "attention_mask": [1] * len(full_ids),
                "record_id": row["id"],
                "category": row.get("category", "uncategorized"),
                "mode": classify_mode(target_content),
            })
    return samples, skipped_long, lengths

TRAIN_SAMPLES, TRAIN_SKIPPED, TRAIN_LENGTHS = expand_assistant_turns(TRAIN_ROWS, "train")
EVAL_SAMPLES, EVAL_SKIPPED, EVAL_LENGTHS = expand_assistant_turns(EVAL_ROWS, "eval")

if len(TRAIN_SAMPLES) < MIN_TRAIN_TURNS or len(EVAL_SAMPLES) < 15:
    raise RuntimeError(
        f"Too few assistant turns after full-prompt tokenization: train={len(TRAIN_SAMPLES)}, eval={len(EVAL_SAMPLES)}. "
        "Use a larger GPU or review the context-length budget; the notebook will not silently truncate the canonical system prompt."
    )
train_skip_ratio = len(TRAIN_SKIPPED) / max(1, train_turns)
if train_skip_ratio > MAX_ALLOWED_SKIP_RATIO:
    raise RuntimeError(
        f"{train_skip_ratio:.1%} of train targets exceed MAX_LENGTH={MAX_LENGTH}. "
        "Use a larger GPU and increase MAX_LENGTH rather than trimming the canonical prompt."
    )

print("Prepared train assistant turns:", len(TRAIN_SAMPLES), "skipped:", len(TRAIN_SKIPPED))
print("Prepared eval assistant turns:", len(EVAL_SAMPLES), "skipped:", len(EVAL_SKIPPED))
print("Train token lengths p50/p90/max:", sorted(TRAIN_LENGTHS)[len(TRAIN_LENGTHS)//2], sorted(TRAIN_LENGTHS)[int(0.9*len(TRAIN_LENGTHS))], max(TRAIN_LENGTHS))
if TRAIN_SKIPPED:
    print("First skipped examples:", TRAIN_SKIPPED[:10])


In [ ]:
TRAIN_DATASET = Dataset.from_list(TRAIN_SAMPLES)
EVAL_DATASET = Dataset.from_list(EVAL_SAMPLES)

class BYVDataCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)
        input_ids, attention_masks, labels = [], [], []
        for item in features:
            pad = max_len - len(item["input_ids"])
            input_ids.append(item["input_ids"] + [self.pad_token_id] * pad)
            attention_masks.append(item["attention_mask"] + [0] * pad)
            labels.append(item["labels"] + [-100] * pad)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

COLLATOR = BYVDataCollator(TOKENIZER.pad_token_id)
print("Assistant-only label masking ready.")


## 7. Load 4-bit base model dan pasang LoRA

T4 menggunakan NF4 + double quantization + FP16 compute. Karena prompt lengkap lebih panjang dibanding dataset sebelumnya, memory probe dan jumlah sample yang terlewati harus diperhatikan.

In [ ]:
MODEL_READY = False
model = None
trainer = None
BASE_MODEL_COMMIT = None

if GPU_AVAILABLE:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    major_version = int(transformers.__version__.split(".")[0])
    dtype_key = "dtype" if major_version >= 5 else "torch_dtype"
    load_kwargs = {
        "revision": MODEL_REVISION,
        "quantization_config": bnb_config,
        "device_map": "auto",
        "low_cpu_mem_usage": True,
    }
    load_kwargs[dtype_key] = COMPUTE_DTYPE
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
    except TypeError as exc:
        if dtype_key == "dtype":
            load_kwargs.pop("dtype", None)
            load_kwargs["torch_dtype"] = COMPUTE_DTYPE
            model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
        else:
            raise exc

    BASE_MODEL_COMMIT = getattr(model.config, "_commit_hash", None)
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora_config)
    for parameter in model.parameters():
        if parameter.requires_grad and parameter.dtype != torch.float32:
            parameter.data = parameter.data.float()

    model.print_trainable_parameters()
    trainable_dtypes = Counter(str(p.dtype) for p in model.parameters() if p.requires_grad)
    print("Trainable parameter dtypes:", dict(trainable_dtypes))
    if any(p.dtype != torch.float32 for p in model.parameters() if p.requires_grad):
        raise RuntimeError("Trainable LoRA parameters are not all FP32; stopping before training.")
    MODEL_READY = True
else:
    print("MODEL_READY=False because CUDA is unavailable.")


## 8. Jalankan SFT QLoRA

In [ ]:
TRAINING_READY = bool(MODEL_READY and len(TRAIN_DATASET) >= MIN_TRAIN_TURNS and len(EVAL_DATASET) >= 15)
if TRAINING_READY:
    from transformers import TrainingArguments, Trainer
    args_signature = inspect.signature(TrainingArguments.__init__).parameters
    eval_key = "eval_strategy" if "eval_strategy" in args_signature else "evaluation_strategy"
    training_kwargs = {
        "output_dir": str(OUTPUT_DIR),
        "run_name": RUN_NAME,
        "num_train_epochs": NUM_EPOCHS,
        "per_device_train_batch_size": PER_DEVICE_BATCH_SIZE,
        "per_device_eval_batch_size": 1,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "warmup_ratio": 0.03,
        "lr_scheduler_type": "cosine",
        "weight_decay": 0.01,
        "optim": "paged_adamw_8bit",
        "logging_steps": 5,
        "save_strategy": "epoch",
        "save_total_limit": 2,
        "fp16": COMPUTE_DTYPE == torch.float16,
        "bf16": COMPUTE_DTYPE == torch.bfloat16,
        "gradient_checkpointing": True,
        "max_grad_norm": 0.3,
        "report_to": "none",
        "remove_unused_columns": False,
        "dataloader_pin_memory": False,
        "seed": SEED,
        "data_seed": SEED,
    }
    training_kwargs[eval_key] = "epoch"
    if "gradient_checkpointing_kwargs" in args_signature:
        training_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}
    training_args = TrainingArguments(**training_kwargs)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=TRAIN_DATASET,
        eval_dataset=EVAL_DATASET,
        data_collator=COLLATOR,
    )
    print("Trainer ready. GPU:", GPU_NAME, "| VRAM:", GPU_VRAM_GB, "GB")
else:
    print("Training not ready. Confirm CUDA, model load, and dataset counts above.")


In [ ]:
if TRAINING_READY:
    try:
        train_result = trainer.train()
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower() or "cuda error" in str(exc).lower():
            torch.cuda.empty_cache()
            raise RuntimeError(
                "CUDA/VRAM failure. Restart runtime, lower MAX_LENGTH only if enough samples still fit, "
                "or use a GPU with more VRAM. Do not truncate the canonical system prompt."
            ) from exc
        raise

    trainer.save_model(str(OUTPUT_DIR))
    TOKENIZER.save_pretrained(str(OUTPUT_DIR))
    metrics = dict(train_result.metrics)
    metrics.update({
        "train_records": len(TRAIN_ROWS),
        "eval_records": len(EVAL_ROWS),
        "train_assistant_turns": len(TRAIN_DATASET),
        "eval_assistant_turns": len(EVAL_DATASET),
        "skipped_long_train_targets": len(TRAIN_SKIPPED),
        "skipped_long_eval_targets": len(EVAL_SKIPPED),
        "base_model": MODEL_ID,
        "base_model_revision_requested": MODEL_REVISION,
        "base_model_commit": BASE_MODEL_COMMIT,
        "dataset_version": DATASET_VERSION,
        "train_sha256": EXPECTED_DATA_SHA256["train.jsonl"],
        "eval_sha256": EXPECTED_DATA_SHA256["eval.jsonl"],
        "prompt_template_git_blob_sha": EXPECTED_PROMPT_GIT_BLOB_SHA,
        "train_modes": dict(train_modes),
        "eval_modes": dict(eval_modes),
        "gpu_name": GPU_NAME,
        "gpu_vram_gb": GPU_VRAM_GB,
        "compute_dtype": str(COMPUTE_DTYPE),
        "max_length": MAX_LENGTH,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
    })
    (EXPERIMENT_DIR / "train_metrics.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8"
    )
    (EXPERIMENT_DIR / "trainer_log_history.json").write_text(
        json.dumps(trainer.state.log_history, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8"
    )
    (EXPERIMENT_DIR / "skipped_long_samples.json").write_text(
        json.dumps({"train": TRAIN_SKIPPED, "eval": EVAL_SKIPPED}, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print("Adapter saved to:", OUTPUT_DIR)
    print(json.dumps(metrics, ensure_ascii=False, indent=2, default=str))
else:
    print("Training skipped because the readiness gate was not satisfied.")


## 9. Smoke test pada eval (bukan held-out test)

Cell ini hanya memakai eval split. Test split tidak dibuka dan tidak dijalankan.

In [ ]:
def generate_text(model_obj, messages, max_new_tokens=MAX_BASELINE_NEW_TOKENS):
    prompt_messages = [dict(m) for m in messages]
    prompt = TOKENIZER.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    device = model_obj.get_input_embeddings().weight.device
    inputs = TOKENIZER(prompt, return_tensors="pt").to(device)
    model_obj.config.use_cache = True
    with torch.inference_mode():
        output = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=TOKENIZER.pad_token_id,
            eos_token_id=TOKENIZER.eos_token_id,
            use_cache=True,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    model_obj.config.use_cache = False
    return TOKENIZER.decode(new_tokens, skip_special_tokens=True).strip()

if TRAINING_READY and trainer is not None:
    smoke_outputs = []
    model.eval()
    for row in EVAL_ROWS[:8]:
        messages = messages_with_system_prompt(row)
        for i, msg in enumerate(messages):
            if msg["role"] != "assistant":
                continue
            pred = generate_text(model, messages[:i])
            record = {
                "id": row["id"],
                "category": row.get("category"),
                "expected_mode": classify_mode(msg["content"]),
                "predicted_mode": classify_mode(pred),
                "prediction": pred,
            }
            smoke_outputs.append(record)
            print("\n=====", row["id"], record["expected_mode"], "=====")
            print(pred[:1200])
    (EXPERIMENT_DIR / "eval_smoke_outputs.jsonl").write_text(
        "".join(json.dumps(x, ensure_ascii=False) + "\n" for x in smoke_outputs), encoding="utf-8"
    )
else:
    print("Smoke test skipped because training did not run.")


## 10. Simpan manifest dan artefak

In [ ]:
import importlib.metadata as metadata
package_versions = {}
for pkg in ["torch", "transformers", "datasets", "accelerate", "peft", "bitsandbytes", "safetensors"]:
    try:
        package_versions[pkg] = metadata.version(pkg)
    except metadata.PackageNotFoundError:
        package_versions[pkg] = None

manifest = {
    "run_name": RUN_NAME,
    "base_model": MODEL_ID,
    "base_model_revision_requested": MODEL_REVISION,
    "base_model_commit": BASE_MODEL_COMMIT,
    "dataset_version": DATASET_VERSION,
    "protocol": "BYV markup v0.1",
    "output_modes": ["TEXT", "MIXED", "STRUCTURED"],
    "method": "SFT + QLoRA",
    "quantization": "4-bit NF4 + double quantization",
    "compute_dtype": str(COMPUTE_DTYPE),
    "thinking_mode": "non-thinking Instruct checkpoint; no think-token injection or hidden reasoning targets",
    "prompt_template_commit": PROMPT_COMMIT,
    "prompt_template_git_blob_sha": EXPECTED_PROMPT_GIT_BLOB_SHA,
    "train_sha256": EXPECTED_DATA_SHA256["train.jsonl"],
    "eval_sha256": EXPECTED_DATA_SHA256["eval.jsonl"],
    "train_records": len(TRAIN_ROWS),
    "eval_records": len(EVAL_ROWS),
    "train_assistant_turns": len(TRAIN_DATASET),
    "eval_assistant_turns": len(EVAL_DATASET),
    "train_modes": dict(train_modes),
    "eval_modes": dict(eval_modes),
    "skipped_long_train_targets": len(TRAIN_SKIPPED),
    "skipped_long_eval_targets": len(EVAL_SKIPPED),
    "test_split_loaded": False,
    "gpu_name": GPU_NAME,
    "gpu_vram_gb": GPU_VRAM_GB,
    "gpu_compute_capability": GPU_COMPUTE_CAPABILITY,
    "max_length": MAX_LENGTH,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "training_completed": bool(TRAINING_READY and trainer is not None and trainer.state.global_step > 0),
    "packages": package_versions,
}
(EXPERIMENT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Manifest:", EXPERIMENT_DIR / "manifest.json")
print("Adapter config exists:", (OUTPUT_DIR / "adapter_config.json").exists())
print("Training test split loaded:", manifest["test_split_loaded"])


## 11. Langkah berikutnya

1. Pastikan `train_metrics.json`, `manifest.json`, dan `eval_smoke_outputs.jsonl` tersimpan di Google Drive.
2. Buka notebook **BYV-AI Test Evaluation BYV Markup v0.1** untuk evaluasi held-out test. Notebook evaluasi membaca `test.jsonl` saja.
3. Review hasil TEXT, MIXED, dan STRUCTURED secara manual, terutama kebenaran isi dan parameter tool.
4. Jangan menyebut model berhasil hanya dari loss atau syntax pass rate. Dataset seed masih perlu human review dan data tambahan.